# IRM vs ERM on Colored-MNIST (Colab T4)
Trustworthy AI — Spurious Correlations & Invariant Learning.
Run All với GPU T4. Full sweep ~10–15 phút; bản quick dưới đây ~3 phút.

In [ ]:
# 0) Bootstrap uv (package manager duy nhất) + deps
!pip install -q uv
!uv pip install --system -q torch torchvision numpy matplotlib seaborn scikit-learn pyyaml tqdm
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# 1) Lấy code (chạy trong repo này thì bỏ qua; trên Colab thì clone)
import os
print(os.getcwd())
# !git clone <your-repo-url> && %cd trustworthy_ai

In [ ]:
# 2) Quick demo: ERM vs IRM vs REx (500 steps, subsample cho nhanh)
import sys; sys.path.insert(0, 'src')
import torch
from cmnist.data import build_colored_mnist, split_environments, experiment_loaders
from cmnist.models import make_model
from cmnist.train import train_one_run
from cmnist.eval import summarize

device = 'cuda' if torch.cuda.is_available() else 'cpu'
envs = split_environments(build_colored_mnist(data_dir='/tmp/data', seed=0))
for method in ['erm', 'irm', 'rex']:
    training, validation, loaders = experiment_loaders(envs, batch_size=2048, seed=0)
    res = train_one_run(lambda: make_model('mlp'),
        training, validation,
        method=method, steps=500, device=device, log_every=500, seed=0)
    s = summarize(res['model'], loaders, device)
    print(f"{method:4s} train={(s['train_e1']['acc']+s['train_e2']['acc'])/2:.3f} "
          f"test={s['test']['acc']:.3f} worst={s['test']['worst']:.3f} ece={s['test']['ece']:.3f}")

In [ ]:
# 3) Full sweep tái hiện (ghi results.json) — chạy khi cần số liệu báo cáo
# !python scripts/run_sweep.py --config configs/base.yaml --out outputs/sweep
# !python scripts/make_plots.py --in outputs/sweep/results.json --out outputs/figs